In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from tqdm import tqdm
import os

# قاموس لتحويل أكواد LCCCode إلى أسماء وصفية بناءً على globcover_LCCS_legend_africa
lcccode_to_name = {
    '0011': 'Bare Areas',
    '6001': 'Rainfed Croplands',
    '6004': 'Sparse Vegetation',
    '11499 // 11500 // 30001': 'Mosaic Cropland And Natural Vegetation',
    '20049 // 20058': 'Bare Areas',
    '42347-R1 // 42347-R2 // 42349-R1 // 42349-R2': 'Artificial Surfaces And Associated Areas'
}

# تحميل الداتا
data = pd.read_csv('climate_data_with_elevation.csv')  # ملف الداتا اللي فيه elevation

# تحميل ملف الـ shapefile لـ land cover
shapefile_path = 'egy_gc_adg.shp'  # الملف في نفس المجلد
landcover_shp = gpd.read_file(shapefile_path)

# طباعة القيم الفريدة في LCCCode للتأكد
print("القيم الفريدة في LCCCode:")
print(landcover_shp['LCCCode'].unique())

# استخراج النقاط الفريدة (LAT, LON)
unique_points = data[['LAT', 'LON']].drop_duplicates().reset_index(drop=True)
print(f"عدد النقاط الفريدة: {len(unique_points)}")

# تحويل النقاط إلى GeoDataFrame
geometry = [Point(xy) for xy in zip(unique_points['LON'], unique_points['LAT'])]
gdf_points = gpd.GeoDataFrame(unique_points, geometry=geometry, crs="EPSG:4326")

# التحقق من وجود ملف التخزين المؤقت
cache_file = 'unique_points_landcover.csv'
if os.path.exists(cache_file):
    print("تحميل أنواع الأرض من ملف التخزين المؤقت...")
    cached_landcover = pd.read_csv(cache_file)
    unique_points = unique_points.merge(
        cached_landcover[['LAT', 'LON', 'land_cover']],
        on=['LAT', 'LON'],
        how='left'
    )
else:
    unique_points['land_cover'] = pd.Series(dtype='object')  # تحديد نوع البيانات كـ object

# دالة لتحديد نوع الأرض
def get_landcover(point, landcover_gdf, lat, lon, max_distance=0.2):  # 0.2 درجة ≈ 22 كم
    """
    تحديد نوع الأرض لنقطة بناءً على الـ shapefile باستخدام LCCCode وتحويله إلى اسم وصفي.
    لو النقطة مش جوا polygon، يتم البحث عن أقرب polygon.
    """
    try:
        # التحقق إذا كانت النقطة جوا polygon
        containing_poly = landcover_gdf[landcover_gdf.contains(point)]
        if not containing_poly.empty:
            lcc_code = containing_poly['LCCCode'].iloc[0]
            return lcccode_to_name.get(lcc_code, 'Unknown')  # تحويل LCCCode إلى اسم وصفي
        
        # تحويل النقاط والـ shapefile إلى نظام مسقط لحساب المسافات
        point_utm = gpd.GeoSeries([point], crs="EPSG:4326").to_crs("EPSG:32636").iloc[0]
        landcover_utm = landcover_gdf.to_crs("EPSG:32636")
        
        # لو مفيش polygon، نبحث عن أقرب واحد
        distances = landcover_utm.distance(point_utm)
        max_distance_meters = max_distance * 111000  # تحويل 0.2 درجة إلى متر (تقريبًا)
        if distances.min() <= max_distance_meters:
            nearest_poly = landcover_gdf.iloc[distances.idxmin()]
            lcc_code = nearest_poly['LCCCode']
            if distances.min() > max_distance_meters * 2:  # تحذير لو المسافة أكتر من 44 كم
                print(f"تحذير: أقرب polygon للنقطة ({lon}, {lat}) بعيد ({distances.min()/1000:.2f} كم)")
            return lcccode_to_name.get(lcc_code, 'Unknown')  # تحويل LCCCode إلى اسم وصفي
        
        # لو مفيش polygon قريب، نرجع Unknown
        print(f"تحذير: لم يتم العثور على نوع أرض للنقطة ({lon}, {lat})")
        return 'Unknown'
    
    except Exception as e:
        print(f"خطأ عند تحديد نوع الأرض للنقطة ({lon}, {lat}): {e}")
        return 'Unknown'

# جلب نوع الأرض للنقاط اللي لسة مالهاش نوع
missing_landcover = unique_points[unique_points['land_cover'].isna()]
if not missing_landcover.empty:
    print(f"جلب أنواع الأرض لـ {len(missing_landcover)} نقاط...")
    landcovers = []
    for index, row in tqdm(missing_landcover.iterrows(), total=len(missing_landcover), desc="جلب أنواع الأرض"):
        landcover = get_landcover(
            gdf_points.loc[index, 'geometry'], 
            landcover_shp, 
            row['LAT'], 
            row['LON']
        )
        landcovers.append(landcover)
    
    # تحديث أنواع الأرض في unique_points
    unique_points.loc[unique_points['land_cover'].isna(), 'land_cover'] = landcovers

# التعامل مع النقطة (35.5, 22.5) لو رجّعت Unknown
missing_point = unique_points[(unique_points['LON'] == 35.5) & (unique_points['LAT'] == 22.5)]
if not missing_point.empty and missing_point['land_cover'].iloc[0] == 'Unknown':
    print("جاري التعامل مع النقطة (35.5, 22.5)...")
    # تحويل النقطة إلى نظام مسقط
    point = gpd.GeoSeries([Point(35.5, 22.5)], crs="EPSG:4326").to_crs("EPSG:32636").iloc[0]
    landcover_utm = landcover_shp.to_crs("EPSG:32636")
    distances = landcover_utm.distance(point)
    nearest_poly = landcover_shp.iloc[distances.idxmin()]
    nearest_landcover = lcccode_to_name.get(nearest_poly['LCCCode'], 'Unknown')
    print(f"أقرب نوع أرض للنقطة (35.5, 22.5): {nearest_landcover}")
    # تحديث كل النقاط اللي ليها نفس الإحداثيات
    unique_points.loc[(unique_points['LON'] == 35.5) & (unique_points['LAT'] == 22.5), 'land_cover'] = nearest_landcover

# حفظ أنواع الأرض في ملف التخزين المؤقت
unique_points[['LAT', 'LON', 'land_cover']].to_csv(cache_file, index=False)
print(f"تم حفظ أنواع الأرض في {cache_file}")

# دمج أنواع الأرض مع الداتا الأصلية
data_with_landcover = data.merge(
    unique_points[['LAT', 'LON', 'land_cover']],
    on=['LAT', 'LON'],
    how='left'
)

# التحقق من وجود قيم فاضية
if data_with_landcover['land_cover'].isna().any():
    print("تحذير: يوجد قيم فاضية في عمود land_cover")
    print(f"عدد القيم الفاضية: {data_with_landcover['land_cover'].isna().sum()}")

# حفظ الداتا في نفس الملف
data_with_landcover.to_csv('climate_data_with_elevation.csv', index=False)
print("تم إضافة عمود 'land_cover' وحفظ الداتا في 'climate_data_with_elevation.csv'")
print(f"عدد القيم الفاضية في عمود land_cover: {data_with_landcover['land_cover'].isna().sum()}")

القيم الفريدة في LCCCode:
['0011' '6001' '7001 // 8001' '11490 // 11494' '0010' '20049 // 20058'
 '0003 / 0004' '21450' '0004 // 0003' '11498' '11491 // 11495'
 '11499 // 11500 // 30001' '6004'
 '42347-R1 // 42347-R2 // 42349-R1 // 42349-R2' '20058' '8005 // 8008'
 '21446 // 21450-121340 / 21454' '0007']
عدد النقاط الفريدة: 90
جلب أنواع الأرض لـ 90 نقاط...


جلب أنواع الأرض:  10%|█         | 9/90 [00:00<00:06, 13.42it/s]

تحذير: لم يتم العثور على نوع أرض للنقطة (35.5, 22.5)


جلب أنواع الأرض: 100%|██████████| 90/90 [00:26<00:00,  3.42it/s]


جاري التعامل مع النقطة (35.5, 22.5)...
أقرب نوع أرض للنقطة (35.5, 22.5): Bare Areas
تم حفظ أنواع الأرض في unique_points_landcover.csv
تم إضافة عمود 'land_cover' وحفظ الداتا في 'climate_data_with_elevation.csv'
عدد القيم الفاضية في عمود land_cover: 0
